# Gravitational Wave Sky Localization Pipeline (Tutorial)

このノートブックでは、重力波（GW）信号の方向推定（Sky Localization）モデルをColab上で構築・学習・推論するエンドツーエンドの手順を解説します。

---

## 1. 環境構築 & リポジトリ取得

GPUが有効になっていることを確認し、リポジトリのクローンと依存パッケージのインストールを行います。

In [ ]:
# GPUの確認
!nvidia-smi

# リポジトリのクローン
!git clone https://github.com/yuya0305-kenkyu/gw-classification.git
%cd gw-classification

# 依存ライブラリのインストール
!pip install -r requirements.txt

##2. 検出器感度データ (PSD) のセットアップ
シミュレーション波形に合成する各干渉計（Advanced LIGO, Advanced Virgo, KAGRA）のPSD感度曲線を準備します。
手動ダウンロード、またはフォールバック用のダミーディレクトリを構築します。

In [ ]:
import os
import numpy as np

os.makedirs('sensitivities', exist_ok=True)

# 実際の感度カーブTXTファイルが存在しない場合、PyCBCの内部モデルへのフォールバックが機能しますが、
# ここでは明示的にダミーASDファイルを配置する例を示します。
freqs = np.linspace(10, 2048, 1000)
dummy_asd = 1e-23 * (freqs / 100.0) ** (-1.5) + 1e-24

for det_file in ['aligo_O4high.txt', 'avirgo_O4high_NEW.txt', 'kagra_128Mpc.txt']:
    path = os.path.join('sensitivities', det_file)
    if not os.path.exists(path):
        np.savetxt(path, np.column_stack([freqs, dummy_asd]))
print("Sensitivities directory configured.")

##3. データの生成 (HDF5)
generate_data.py を使用して、学習データとテストデータを生成します。Colabのメモリと時間を考慮し、サンプルサイズを調整可能です。

In [ ]:
# クイック動作用のサンプルデータ生成
!python data_generation/generate_data.py --mode sample --output_file sample_data.hdf5

# 本格検証用のテストデータ生成 (例: 500サンプル)
!python data_generation/generate_data.py --mode test --num_samples 500 --output_file test_data.hdf5

# 訓練データ生成 (例: 2000サンプル)
!python data_generation/generate_data.py --mode train --num_samples 2000 --output_file training_data.hdf5

##4. モデルの学習 (MLP / TCN)
train.py でモデルの訓練を開始します。特徴量ベースの全結合ネットワーク（MLP: --method 1）または時系列波形直接入力（TCN: --method 2）を選択します。

In [ ]:
# MLP の学習例 (軽量・高速)
!python train.py \
    --hdf_file training_data.hdf5 \
    --method 1 \
    --epoch 30 \
    --batch_size 128 \
    --training_size 1600 \
    --weights_directory weights

In [ ]:
# TCN の学習例 (時系列畳み込み)
!python train.py \
    --hdf_file training_data.hdf5 \
    --method 2 \
    --epoch 30 \
    --batch_size 64 \
    --training_size 1600 \
    --weights_directory weights

##5. 学習曲線の可視化
保存された学習ログCSVをもとに、損失値（Loss）と精度（Accuracy）の推移をプロットします。

In [ ]:
%matplotlib inline
import glob
from plot_history import plot_learning_curve

csv_logs = glob.glob('weights/history_*.csv')
if csv_logs:
    print(f"Plotting: {csv_logs[0]}")
    plot_learning_curve(csv_logs[0])
else:
    print("No history CSV found in weights/.")

##6. テストデータでの評価・推論
訓練済みの重みを用いてテストセットに対する天空領域分類精度を算出します。アンサンブル推論（--method 3）も可能です。

In [ ]:
# MLPによる推論
!python predict.py \
    --hdf_file test_data.hdf5 \
    --method 1 \
    --weights_directory weights